In [3]:
!pip install -r {Path(os.getcwd()).parent / "requirements.txt"}
!pip install opencv-python matplotlib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 6.2 MB/s  0:01:09m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 8.8 MB/s  0:00:43m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.8 MB/s  0:00:21m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 8.9 MB/s  0:00:23m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 8.1 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 8.8 MB/s  0:00:22m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 7.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 6.1 MB/s  0:00:57m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 9.1 MB/s  0:00:01eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 8.9 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 7.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [42]:
import os
import sys
import json
import yaml
import time
import warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import torch
import cv2
from ultralytics import YOLO, RTDETR
from utils.logger import ModelLogger, save_results
warnings.filterwarnings('ignore')

In [44]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [46]:
PROJECT_ROOT = Path("../").resolve()

DATA_ROOT = PROJECT_ROOT / "data" / "data"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YAML_PATH = PROJECT_ROOT / "configs" / "dataset_config.yaml"

MODELS = {
    "YOLOv8n": PROJECT_ROOT / "models/weights/trained/yolo_n/best.pt",
    "YOLOv8s": PROJECT_ROOT / "models/weights/trained/yolo_s/best.pt",
    "YOLOv8m": PROJECT_ROOT / "models/weights/trained/yolo_m/best.pt",
    "RT-DETR": PROJECT_ROOT / "models/weights/trained/rtdetr/best.pt",
    "Faster R-CNN": PROJECT_ROOT / "models/weights/trained/faster_rcnn/faster_rcnn/best.pth"
}

In [47]:
dataset_yaml = {
    "path": str(DATA_ROOT),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 1,
    "names": ["license_plate"]
}

with open(YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        dataset_yaml,
        f,
        sort_keys=False,
        allow_unicode=True
    )

In [48]:
print("train:", (DATA_ROOT / "train/images").exists())
print("val:", (DATA_ROOT / "val/images").exists())
print("test:", (DATA_ROOT / "test/images").exists())

assert (DATA_ROOT / "train/images").exists()
assert (DATA_ROOT / "val/images").exists()
assert (DATA_ROOT / "test/images").exists()

train: True
val: True
test: True


In [49]:
def evaluate_model(model_name, model_path):

    print(f"\n{'='*60}")
    print(model_name)
    print('='*60)

    if "RT-DETR" in model_name:
        model = RTDETR(str(model_path))
    else:
        model = YOLO(str(model_path))

    start = time.time()

    results = model.val(
        data=str(YAML_PATH),
        split="test",
        plots=True,
        save_json=True,
        verbose=False
    )

    total_time = time.time() - start

    precision = float(results.box.mp)
    recall = float(results.box.mr)

    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    metrics = {
        "Model": model_name,
        "mAP50": float(results.box.map50),
        "mAP50_95": float(results.box.map),
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Inference_ms": float(results.speed["inference"]),
        "Preprocess_ms": float(results.speed["preprocess"]),
        "Postprocess_ms": float(results.speed["postprocess"]),
        "Validation_time_sec": total_time,
        "Model_size_MB": round(
            Path(model_path).stat().st_size / 1024 / 1024,
            2
        )
    }

    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")

    return metrics

In [50]:
all_results = []

for name, path in MODELS.items():

    if not path.exists():
        print(f"Missing: {path}")
        continue

    metrics = evaluate_model(name, path)

    all_results.append(metrics)


YOLOv8n
Ultralytics 8.4.70 🚀 Python-3.12.11 torch-2.12.1+cu130 CPU (Intel Core i7-7700K 4.20GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 3.2±1.1 ms, read: 11.4±7.4 MB/s, size: 316.2 KB)
val: Scanning /home/jovyan/work/data/data/test/labels.cache... 769 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 769/769 215.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 5.7s/it 4:376.6ss
                   all        769        809      0.983      0.974      0.992      0.818
Speed: 3.9ms preprocess, 317.8ms inference, 0.0ms loss, 3.1ms postprocess per image
Saving /home/jovyan/work/notebooks/runs/detect/val-9/predictions.json...
Results saved to /home/jovyan/work/notebooks/runs/detect/val-9
mAP50: 0.9919
mAP50_95: 0.8177
Precision: 0.9829
Recall: 0.9740
F1: 0.9784
Inference_ms: 317.7527
Preprocess_ms: 3.8896
Postprocess_ms: 3.1389
Validation_tim

AssertionError: /home/jovyan/work/models/weights/trained/faster_rcnn/faster_rcnn/best.pth acceptable suffix is {'.pt'}, not .pth

In [ ]:
results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    by="mAP50",
    ascending=False
)

results_df

In [ ]:
save_results(
    results_df, 
    OUTPUT_DIR,
    metadata={
        'dataset': str(YAML_PATH),
        'num_images': 769,
        'num_classes': 1
    }
)